# Dose and dose-rate estimation — v1

Sam Tardif (samuel.tardif@cea.fr), Thibaut Jousseaume

This notebook accompanies *How beam damage can skew synchrotron operando studies of batteries*, ACS Energy Lett. 8, 3323 (2023), [doi:10.1021/acsenergylett.3c00815](https://doi.org/10.1021/acsenergylett.3c00815).

**What changed in v1** (see `CHANGELOG.md`):

* the deposited energy uses the NIST **mass energy-absorption coefficient** μ_en/ρ; the beam is still attenuated with the mass attenuation coefficient μ/ρ (also NIST, now including Rayleigh scattering);
* spectra (white/pink beams, harmonics), Gaussian beams, tilted samples, depth profiles, mixtures and solutions, exposure histories, and an optional photoelectron-escape estimate for micro-beams;
* the 2023 API (`Beam(beamEnergy=...)`, `Layer(solidMaterial=...)`, `Stacking`) still works; section 1 is the 2023 cell unchanged.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from dose_estimation import (Beam, Spectrum, Material, Layer, Stack, Stacking,
                             ExposureSchedule)
from dose_estimation.plotting import plot_depth_profiles

## 1. The 2023 cell, unchanged (v0 API)

This is the cell of the paper's SI-10 (pouch cell c, location #3, 3 h at C/8). It now runs with μ_en and emits a `FutureWarning` once. `get_dose(..., deposition="attenuation")` reproduces the 2023 model (with NIST μ, 3–10 % above the Henke + Klein-Nishina values used then).

In [ ]:
#-------------------------------------------------------------------------------------------------
# X-RAY BEAM: beamEnergy in eV, photonFlux in photon/s, beamWidth and beamHeight in um, exposure in s
xr = Beam(beamEnergy = 27000,
          photonFlux = 2.44E10,
          beamWidth = 630,
          beamHeight = 110,
          exposure = 1050*27*.375)
#-------------------------------------------------------------------------------------------------
# LAYERS: density in g/cm3, thickness in mm, porosity unitless (0 <= porosity <= 1)
current_collector_anode = Layer(name = "Cu collector",
                                solidMaterial = Material(name = "copper", formula = "Cu", density = 8.96),
                                liquidMaterial = None, thickness = 0.010, porosity = 0.0)
anode = Layer(name = "anode",
              solidMaterial = Material(name = "graphite", formula = "C", density = 2.3),
              liquidMaterial = Material(name = "electrolyte", formula = "C3H6O3", density = 1.15),
              thickness = 0.054, porosity = 0.40)
separator = Layer(name = "separator",
                  solidMaterial = Material(name = "glass fiber", formula = "SiO2", density = 2.6),
                  liquidMaterial = Material(name = "electrolyte", formula = "C3H6O3", density = 1.15),
                  thickness = 0.025, porosity = 0.90)
cathode = Layer(name = "cathode",
                solidMaterial = Material(name = "NMC", formula = "LiNi0.8Mn0.1Co0.1O2", density = 4.8),
                liquidMaterial = Material(name = "electrolyte", formula = "C3H6O3", density = 1.15),
                thickness = 0.032, porosity = 0.28)
current_collector_cathode = Layer(name = "Al collector",
                                  solidMaterial = Material(name = "aluminium", formula = "Al", density = 2.7),
                                  liquidMaterial = None, thickness = 0.012, porosity = 0.0)
# BUILD THE CELL (first the component where the beam enters)
pouch_cell = Stacking([current_collector_anode, anode, separator, cathode, current_collector_cathode])
pouch_cell.get_dose(xr)
fig, ax = pouch_cell.get_visual_dose()

In [ ]:
# The v0 attributes are still there ...
print(f"NMC dose (mu_en):        {cathode.solidMaterial.dose_kGy:8.0f} kGy")
# ... and the 2023 model is one keyword away
pouch_cell.get_dose(xr, deposition="attenuation")
print(f"NMC dose (attenuation):  {cathode.solidMaterial.dose_kGy:8.0f} kGy  (2023 notebook: 10661 kGy)")

## 2. The same cell with the v1 API

Thicknesses are in µm, energies in eV, results come back in a `DoseResult` instead of being written onto the materials, so one electrolyte instance can be used in every layer. Each component gets:

* `dose_rate`: depth-averaged, profile-mean dose rate (the 2023 quantity);
* `dose_rate_max`: at the entrance face and the beam centre;
* `mixed_dose_rate` of its layer: the limit where secondary electrons are shared by all phases (electron range ≫ particle size). The true dose of a minor phase (electrolyte, binder) next to a strong absorber lies between its own value and this one.

In [ ]:
copper = Material("Cu", 8.96, name="copper")
graphite = Material("C", 2.3, name="graphite")
glass = Material("SiO2", 2.6, name="glass fiber")
nmc = Material("LiNi0.8Mn0.1Co0.1O2", 4.8, name="NMC811")
alu = Material("Al", 2.7, name="aluminium")
dmc = Material("C3H6O3", 1.15, name="electrolyte")     # one instance, reused

cell_2023 = Stack([
    Layer.solid("Cu collector", copper, thickness_um=10),
    Layer.porous("anode", graphite, porosity=0.40, filler=dmc, thickness_um=54),
    Layer.porous("separator", glass, porosity=0.90, filler=dmc, thickness_um=25),
    Layer.porous("cathode", nmc, porosity=0.28, filler=dmc, thickness_um=32),
    Layer.solid("Al collector", alu, thickness_um=12),
])
beam = Beam.monochromatic(27e3, 2.44e10, width_um=630, height_um=110)
exposure = 1050 * 27 * 0.375          # s, location #3 after 3 h at C/8

res = cell_2023.compute(beam, exposure_s=exposure)
res_v0 = cell_2023.compute(beam, exposure_s=exposure, deposition="attenuation")   # 2023 model
tab = res.table("kGy")
tab["dose_attenuation_kGy"] = res_v0.table("kGy")["dose_kGy"]     # 2023 model with NIST mu
tab["ratio"] = tab["dose_kGy"] / tab["dose_attenuation_kGy"]
tab[["layer", "material", "dose_attenuation_kGy", "dose_kGy", "ratio", "dose_max_kGy"]].round(3)

In [ ]:
ax = res.plot(what="dose", unit="kGy")
plt.tight_layout()

## 3. A stack closer to the cell of the paper

SI-1 and Table S1 describe the cell more precisely than the 2023 notebook did:

* electrolyte: 1 M LiPF6 in EC:PC:DMC 1:1:3 (v/v) — the P and F of the salt raise μ_en/ρ by ~35 % at 27 keV;
* separator: 25 µm Celgard (polyolefin), not glass fibre;
* cathode coating: NMC811 / PVDF / C65 / VGCF = 96 / 2.5 / 1 / 0.5 wt%, 26 % porosity, on 20 µm Al;
* the cell sits between Si wafers and inside a pouch.

Values marked **assumed** below are not given in the paper; replace them with the real ones. If the 2.44×10¹⁰ ph/s already accounts for the upstream window and pouch, set `si_window_um = 0` and `pouch_al_um = 0`.

In [ ]:
# Electrolyte: solvent blend by volume, then 1 M LiPF6
ec = Material("C3H4O3", 1.32, name="EC")
pc = Material("C4H6O3", 1.205, name="PC")
dmc_pure = Material("C3H6O3", 1.07, name="DMC")
solvent = Material.from_volume_fractions({ec: 1, pc: 1, dmc_pure: 3})
electrolyte = Material.solution(solvent, "LiPF6", molarity=1.0, density=1.27,   # density assumed
                                name="1M LiPF6 EC:PC:DMC")
print(solvent.describe()); print(electrolyte.describe())

# Cathode solids: NMC811 + PVDF + carbon (96/2.5/1.5 wt%) -> volume fractions in the layer
pvdf = Material("C2H2F2", 1.78, name="PVDF")
carbon = Material("C", 1.9, name="carbon black")       # assumed density
w = {nmc: 0.96, pvdf: 0.025, carbon: 0.015}
v_solid = {m: wi / m.density for m, wi in w.items()}
norm = sum(v_solid.values())
porosity_cathode = 0.26
cathode_components = [(m, (1 - porosity_cathode) * v / norm) for m, v in v_solid.items()]
cathode_components.append((electrolyte, porosity_cathode))

celgard = Material("C3H6", 0.90, name="Celgard (PP)")
silicon = Material("Si", 2.33, name="Si wafer")
si_window_um = 0       # assumed: set the wafer thickness if it is upstream of where the flux was measured
pouch_al_um = 0        # assumed: Al layer of the pouch laminate

layers = []
if si_window_um:
    layers.append(Layer.solid("Si window", silicon, si_window_um))
if pouch_al_um:
    layers.append(Layer.solid("pouch Al", alu, pouch_al_um))
layers += [
    Layer.solid("Cu collector", copper, 10),
    Layer.porous("anode", graphite, porosity=0.40, filler=electrolyte, thickness_um=54),
    Layer.porous("separator", celgard, porosity=0.40, filler=electrolyte, thickness_um=25),  # porosity assumed
    Layer("cathode", components=cathode_components, thickness_um=32),
    Layer.solid("Al collector", alu, 20),
]
cell = Stack(layers)
res_cell = cell.compute(beam, exposure_s=exposure)
print(res_cell.summary("MGy"))

## 4. Beam profile, depth and exposure history

* A Gaussian beam of the same FWHM has a peak fluence 0.88 N/(wh) and a signal-weighted mean 0.44 N/(wh), versus N/(wh) for the top-hat convention used in 2023.
* `depth_profile` shows the dose through the stack (the Cu foil and the NMC layer shade what lies behind them).
* `ExposureSchedule` turns a dose rate into a dose history: location #3 was measured 1050 s at a time, while the beam visited the other two locations.

In [ ]:
gauss = Beam.monochromatic(27e3, 2.44e10, width_um=630, height_um=110, profile="gaussian")
rg = cell.compute(gauss, exposure_s=exposure)
c_top, c_gau = res_cell.get("cathode", "NMC811"), rg.get("cathode", "NMC811")
print(f"NMC811 top-hat: {c_top.dose/1e6:.2f} MGy;  Gaussian: mean {c_gau.dose/1e6:.2f} MGy, peak {c_gau.dose_max/1e6:.2f} MGy")

ax = plot_depth_profiles(res_cell, unit="kGy")
ax.set_title("dose rate through the cell (top-hat beam)", pad=14)

In [ ]:
rate = res_cell.get("cathode", "NMC811").dose_rate                 # Gy/s while the beam is on
# Location #3: 1050 s exposures, assumed one visit every 1150 s (the other spots took ~100 s)
intermittent = ExposureSchedule.periodic(on_s=1050, period_s=1150, n=10)
continuous = ExposureSchedule.continuous(intermittent.total_exposure_s)
fig, ax = plt.subplots(figsize=(7, 3.5))
for s, lab in [(intermittent, "intermittent (loc. #3)"), (continuous, "continuous, same exposure")]:
    t, d = s.dose_history(rate)
    ax.plot(t / 3600, d / 1e6, label=f"{lab}: mean rate {s.mean_dose_rate(rate):.0f} Gy/s")
ax.axhline(1e6 * 1e-6, color="0.6", ls=":"); ax.text(0.05, 1.05, "1 MGy", color="0.4")
ax.set_xlabel("time (h)"); ax.set_ylabel("NMC811 dose (MGy)"); ax.legend(fontsize="small")

## 5. Why μ_en matters more at high energy and for light phases

The energy deposited per incident photon and per unit fluence is E·μ_en/ρ. For carbon and organic electrolytes, Compton scattering takes over above ~20 keV and most of the interacting energy leaves with the scattered photon, so the 2023 model (E·μ/ρ) overestimates their dose increasingly with energy.

In [ ]:
energies = np.geomspace(5e3, 100e3, 200)
fig, ax = plt.subplots(figsize=(7, 4))
for m, col in [(nmc, "C0"), (electrolyte, "C1"), (graphite, "C2"), (celgard, "C3")]:
    ax.plot(energies / 1e3, m.mu_en_rho(energies) / m.mu_rho(energies), color=col, label=m.name)
ax.axvline(27, color="0.7", ls=":")
ax.set_xscale("log"); ax.set_xlabel("photon energy (keV)"); ax.set_ylabel("μ_en / μ  (= corrected / 2023 dose)")
ax.legend(fontsize="small"); ax.set_ylim(0, 1.05)